# Module 4: A/B Testing Framework & Statistical Inference

## 1. Experimental Foundations in Digital Content

Traditional social media content evaluation often relies on superficial observational comparisons: creators compare two posts published at different times, with different topics, and claim that a higher like count proves that one hook caused better performance.

In this module, we implement a **statistically defensible experimentation engine** distinguishing:
1. **Randomized Controlled Trials (RCTs):** Controlled audience splits with balanced exposure.
2. **Observational Cohort Comparisons:** Content comparisons where confounders (time of day, topic, algorithmic distribution) are not randomized, requiring explicit non-causal labeling.

> **Dataset Provenance Notice:** This notebook uses curated synthetic benchmark experiment fixtures ($N=4$ experiments, $N=380$ observations) to demonstrate hypothesis testing, multiple comparison adjustments, and power calculations.

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
from src.statistics.ab_testing import ABTestingFramework, ComparisonResult
from src.data_ingestion.schemas import ExperimentType, ExperimentStatus

framework = ABTestingFramework(alpha=0.05)
df_experiments = pd.read_csv("../data/sample/sample_experiments.csv")
df_observations = pd.read_parquet("../data/processed/processed_ab_tests.parquet")

print(f"Loaded {len(df_experiments)} experiments and {len(df_observations)} observations.")
display(df_experiments)

## 2. Proportion Metrics & Experimental Units

For conversion/rate metrics (e.g. `share_rate` = shares / reach):
- **Aggregate Exposure Unit:** Under H0: p_A = p_B, two-proportion pooled z-test is evaluated. When expected or observed cells are sparse (< 5), Fisher's exact test serves as the primary inferential fallback controlling the verdict.
- **Agresti–Caffo adjusted Wald interval**, which generally improves coverage compared with the conventional Wald interval, particularly for small samples and boundary proportions. Coverage remains approximate.
- **Effect Size:** Quantified via Cohen's h: 2 * (arcsin(sqrt(p_B)) - arcsin(sqrt(p_A))).
- **Clustering Defense:** In multi-post creative experiments, aggregate exposure proportions ignore intra-post clustering (ICC > 0). The cluster-defensible post-level approach (`compare_post_level_rates`) collapses exposures into independent post-level rates.


In [ ]:
exp1_obs = df_observations[df_observations["experiment_id"] == "EXP_001"]
sub_a = exp1_obs[exp1_obs["variant"] == "Story Hook"]
sub_b = exp1_obs[exp1_obs["variant"] == "Visual Hook"]

shares_a, reach_a = int(sub_a["shares"].sum()), int(sub_a["reach"].sum())
shares_b, reach_b = int(sub_b["shares"].sum()), int(sub_b["reach"].sum())

res_prop = framework.compare_proportions(
    count_a=shares_a, nobs_a=reach_a,
    count_b=shares_b, nobs_b=reach_b,
    metric_name="share_rate",
    variant_a_label="Story Hook",
    variant_b_label="Visual Hook"
)

print(f"Control Share Rate (Story Hook): {res_prop.value_a:.4%}")
print(f"Treatment Share Rate (Visual Hook): {res_prop.value_b:.4%}")
print(f"Relative Lift: {res_prop.relative_lift_pct:+.2f}%")
print(f"z-statistic: {res_prop.test_statistic:.4f} | p-value: {res_prop.p_value:.5e}")
print(f"Cohen's h effect size: {res_prop.effect_size:.4f}")
print(f"95% Confidence Interval for Difference: [{res_prop.confidence_interval[0]:.4f}, {res_prop.confidence_interval[1]:.4f}]")
print(f"Verdict: {res_prop.verdict}")

## 3. Continuous Metrics: Welch's t-test and Mann-Whitney U

For continuous outcomes (e.g. `views`, `retention_rate`, `relatability_pct`):
- We use **Welch's t-test** (`equal_var=False`), which does not assume equal population variances.
- We evaluate non-parametric **Mann-Whitney U** tests to handle viral skewness.
- Effect size is quantified via **Hedges' $g$** (small-sample corrected Cohen's $d$).

In [ ]:
exp2_obs = df_observations[df_observations["experiment_id"] == "EXP_002"]
sample_a = exp2_obs[exp2_obs["variant"] == "Question Hook"]["views"].values
sample_b = exp2_obs[exp2_obs["variant"] == "Bold Statement Hook"]["views"].values

res_cont = framework.compare_continuous(
    sample_a, sample_b,
    metric_name="views",
    variant_a_label="Question Hook",
    variant_b_label="Bold Statement Hook"
)

print(f"Control Mean Views: {res_cont.value_a:,.2f}")
print(f"Treatment Mean Views: {res_cont.value_b:,.2f}")
print(f"Relative Lift: {res_cont.relative_lift_pct:+.2f}%")
print(f"Welch t-stat: {res_cont.test_statistic:.4f} | p-value: {res_cont.p_value:.5e}")
print(f"Mann-Whitney p-value: {res_cont.mann_whitney_p_value:.5e}")
print(f"Hedges' g effect size: {res_cont.effect_size:.4f}")
print(f"95% Confidence Interval: [{res_cont.confidence_interval[0]:,.2f}, {res_cont.confidence_interval[1]:,.2f}]")
print(f"Verdict: {res_cont.verdict}")

## 4. Multiple Hypotheses Testing: Holm-Bonferroni vs Benjamini-Hochberg

When evaluating multiple metrics simultaneously, false positive risk inflates: $\text{FWER} = 1 - (1 - \alpha)^m$.
We test and compare Holm-Bonferroni (FWER control) vs Benjamini-Hochberg (FDR control).

In [ ]:
metrics_to_test = ["retention_rate", "views", "shares", "saves", "relatability_pct"]
raw_tests = []
for m in metrics_to_test:
    s_a = exp1_obs[exp1_obs["variant"] == "Story Hook"][m].dropna().values
    s_b = exp1_obs[exp1_obs["variant"] == "Visual Hook"][m].dropna().values
    raw_tests.append(framework.compare_continuous(s_a, s_b, metric_name=m, variant_a_label="Story Hook", variant_b_label="Visual Hook"))

holm_corrected = ABTestingFramework.apply_multiple_testing_correction(raw_tests, method="holm")

rows = []
for r in holm_corrected:
    rows.append({
        "Metric": r.metric_name,
        "Raw p-value": f"{r.p_value:.5f}",
        "Adjusted p-value (Holm)": f"{r.p_value_adjusted:.5f}",
        "Significant (Adj)": r.is_statistically_significant
    })
display(pd.DataFrame(rows))

## 5. Sample Size and Power Planning

Pre-experiment sample size estimation ensures tests are designed with adequate statistical power ($1 - \beta = 80\%$).

In [ ]:
plan_prop = ABTestingFramework.calculate_sample_size_proportions(
    baseline_rate=0.03,
    mde_relative=0.20,
    alpha=0.05,
    power=0.80
)
print("Sample Size Planning for Proportion Metric:")
for k, v in plan_prop.items():
    print(f"  {k}: {v}")

plan_cont = ABTestingFramework.calculate_sample_size_continuous(
    baseline_mean=100.0,
    baseline_std=25.0,
    mde_relative=0.10,
    alpha=0.05,
    power=0.80
)
print("\nSample Size Planning for Continuous Metric:")
for k, v in plan_cont.items():
    print(f"  {k}: {v}")